In [46]:
!pip install --upgrade transformers datasets evaluate peft accelerate bitsandbytes rouge_score -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 637.4/637.4 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 100.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 115.4 MB/s eta 0:00:00


In [47]:
!pip uninstall -y transformers
!pip install transformers==4.41.2 -q

Found existing installation: transformers 5.5.0
Uninstalling transformers-5.5.0:
  Successfully uninstalled transformers-5.5.0


In [49]:
import torch
from datasets import load_dataset
from transformers import T5Tokenizer, T5ForConditionalGeneration, Trainer, TrainingArguments
import evaluate
import pandas as pd

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cuda


In [50]:
from datasets import load_dataset

# 🔥 SET SAMPLE SIZE
MAX_SAMPLES = 1500

# Load dataset
dataset = load_dataset("cnn_dailymail", "3.0.0")

# Format into input-output pairs
def format_data(example):
    return {
        "input": "summarize: " + example["article"],
        "output": example["highlights"]
    }

# 🔥 TAKE 1500 SAMPLES
dataset = dataset["train"].shuffle(seed=42).select(range(MAX_SAMPLES))

# Apply formatting
dataset = dataset.map(format_data)

# Split into train/test
dataset = dataset.train_test_split(test_size=0.1)

print(dataset)

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['article', 'highlights', 'id', 'input', 'output'],
        num_rows: 1350
    })
    test: Dataset({
        features: ['article', 'highlights', 'id', 'input', 'output'],
        num_rows: 150
    })
})


In [51]:
model_name = "t5-base"
tokenizer = T5Tokenizer.from_pretrained(model_name)

def load_base_model():
    return T5ForConditionalGeneration.from_pretrained(model_name).to(device)

In [52]:
def tokenize(example):
    inputs = tokenizer(example["input"], truncation=True, padding="max_length", max_length=512)
    targets = tokenizer(example["output"], truncation=True, padding="max_length", max_length=128)
    inputs["labels"] = targets["input_ids"]
    return inputs

tokenized = dataset.map(tokenize, batched=True)

Map:   0%|          | 0/1350 [00:00<?, ? examples/s]

Map:   0%|          | 0/150 [00:00<?, ? examples/s]

In [53]:
def train_model(model, output_dir, lr=5e-5, batch_size=8, steps=600):

    args = TrainingArguments(
        output_dir=output_dir,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,

        max_steps=steps,   # 🔥 fixed steps for fair comparison

        learning_rate=lr,
        save_strategy="no",
        logging_steps=20,

        fp16=True,
        gradient_accumulation_steps=2,

        warmup_steps=50,        # 🔥 helps stability
        weight_decay=0.01,      # 🔥 prevents overfitting
        report_to="none"        # 🔥 avoids warnings in colab
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=tokenized["train"],
        eval_dataset=tokenized["test"]
    )

    trainer.train()
    return model

In [54]:
def generate_summary(model, text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=80,
        num_beams=4,
        no_repeat_ngram_size=3,
        repetition_penalty=1.2,
        length_penalty=1.0,
        early_stopping=True,
        min_length=30,
        do_sample=False
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
import evaluate

# Load ROUGE metric
rouge = evaluate.load("rouge")

def evaluate_model(model, samples=100):
    model.eval()   # 🔥 important

    preds = []
    refs = []

    for sample in dataset["test"].select(range(samples)):
        input_text = sample["input"]
        reference = sample["output"]

        prediction = generate_summary(model, input_text)

        preds.append(prediction)
        refs.append(reference)

    scores = rouge.compute(
        predictions=preds,
        references=refs
    )

    return scores

In [55]:
base_model = load_base_model()

# 🔥 Evaluate on more samples for stability
base_results = evaluate_model(base_model, samples=100)

print("\n📊 Base Model Results:")
for k, v in base_results.items():
    print(f"{k}: {round(v, 4)}")

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]


📊 Base Model Results:
rouge1: 0.3902
rouge2: 0.1976
rougeL: 0.2902
rougeLsum: 0.342


In [56]:
# 🔥 Full Fine-Tuning (with more training)
model_full = load_base_model()

model_full = train_model(
    model_full,
    "full_ft",
    lr=3e-5,
    batch_size=8,
    steps=900   # ✅ THIS is what you wanted
)

full_results = evaluate_model(model_full)

print("\n📊 Full FT Results:")
for k, v in full_results.items():
    print(f"{k}: {round(v, 4)}")

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

Step,Training Loss
20,16.390080
40,10.786629
60,4.093631
80,2.567409
100,2.071944
120,2.017521
140,2.014448
160,1.885817
180,1.792684
200,1.795510



📊 Full FT Results:
rouge1: 0.4165
rouge2: 0.2142
rougeL: 0.3058
rougeLsum: 0.3633


In [57]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q", "v"],   # for T5
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_2_SEQ_LM"
)

In [58]:
from peft import LoraConfig, get_peft_model

model_lora = load_base_model()
model_lora = get_peft_model(model_lora, lora_config)

model_lora = train_model(
    model_lora,
    "lora_ft",
    lr=2e-4,
    batch_size=8,
    steps=700
)

lora_results = evaluate_model(model_lora)

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

Step,Training Loss
20,16.908794
40,15.716217
60,11.945609
80,4.903983
100,2.724872
120,2.537379
140,2.315726
160,2.081009
180,1.983121
200,1.977316


In [63]:
lora_results = evaluate_model(model_lora)

print("\n📊 Full FT Results:")
for k, v in lora_results.items():
    print(f"{k}: {round(v, 4)}")


📊 Full FT Results:
rouge1: 0.4142
rouge2: 0.2046
rougeL: 0.2954
rougeLsum: 0.357


In [67]:
# Hyperparameter Fine-Tuning
# Different learning rate, steps, batch size
model_hp = load_base_model()

model_hp = train_model(
    model_hp,
    output_dir="hp_ft",
    lr=1e-4,        # higher learning rate
    batch_size=8,
    steps=900       # similar steps as full FT
)

hp_results = evaluate_model(model_hp)
print("HP Fine-Tuning Results:", hp_results)

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

Step,Training Loss
20,14.384297
40,4.278327
60,2.162352
80,1.970604
100,1.723205
120,1.726286
140,1.812387
160,1.726870
180,1.595797
200,1.554455


HP Fine-Tuning Results: {'rouge1': np.float64(0.4169538248541247), 'rouge2': np.float64(0.218742797562596), 'rougeL': np.float64(0.3155616492847343), 'rougeLsum': np.float64(0.36766086885318605)}


In [68]:
comparison = pd.DataFrame({
    "Metric": ["ROUGE-1", "ROUGE-2", "ROUGE-L"],
    "Base Model": [
        base_results["rouge1"],
        base_results["rouge2"],
        base_results["rougeL"]
    ],
    "Full FT": [
        full_results["rouge1"],
        full_results["rouge2"],
        full_results["rougeL"]
    ],
    "LoRA": [
        lora_results["rouge1"],
        lora_results["rouge2"],
        lora_results["rougeL"]
    ],
    "HP-FT": [
        hp_results["rouge1"],
        hp_results["rouge2"],
        hp_results["rougeL"]
    ]
}).round(4)

comparison

,Metric,Base Model,Full FT,LoRA,HP-FT
0,ROUGE-1,0.3902,0.4165,0.4142,0.4170
1,ROUGE-2,0.1976,0.2142,0.2046,0.2187
2,ROUGE-L,0.2902,0.3058,0.2954,0.3156


In [69]:
def test_examples(samples=3):
    for i, sample in enumerate(dataset["test"].select(range(samples))):
        print(f"\n=== SAMPLE {i+1} ===")
        print("INPUT:", sample["article"][:200], "...")
        print("REFERENCE:", sample["highlights"])
        print("BASE:", generate_summary(base_model, sample["article"]))
        print("FULL FT:", generate_summary(model_full, sample["article"]))
        print("LoRA:", generate_summary(model_lora, sample["article"]))
        print("HP-FT:", generate_summary(model_hp, sample["article"]))
        print("="*80)

test_examples(3)


=== SAMPLE 1 ===
INPUT: By . Daily Mail Reporter . PUBLISHED: . 17:31 EST, 9 January 2013 . | . UPDATED: . 17:31 EST, 9 January 2013 . American officials blamed the Iranian government for a series of coordinated hacking atta ...
REFERENCE: American banks targeted in string of disruptive hacking attacks .
Because no money was stolen, officials believe it was a coordinated attack approved by the Iranian government, though they deny any connection .
Group claims responsibility, saying they did it because controversial 'Innocence of Muslims' film is still available online .
BASE: a series of coordinated cyber attacks have disrupted service to the data centers for a long list of U.S.-based banks . officials believe that the hacks were approved by the Iranian government, headed by Mahmoud Ahmadinejad . the hackers targeted bank databases and flooded their systems causing major disruptions and even complete shutdowns of company networks 
FULL FT: American officials blame the Iranian governme

In [70]:
model_full.save_pretrained("full_ft_model")
model_lora.save_pretrained("lora_model")
model_hp.save_pretrained("hp_ft_model")
base_model.save_pretrained("base_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [71]:
!zip -r all_models.zip base_model full_ft_model lora_model hp_ft_model

  adding: base_model/ (stored 0%)
  adding: base_model/model.safetensors (deflated 53%)
  adding: base_model/generation_config.json (deflated 29%)
  adding: base_model/config.json (deflated 63%)
  adding: full_ft_model/ (stored 0%)
  adding: full_ft_model/model.safetensors (deflated 8%)
  adding: full_ft_model/generation_config.json (deflated 29%)
  adding: full_ft_model/config.json (deflated 63%)
  adding: lora_model/ (stored 0%)
  adding: lora_model/adapter_config.json (deflated 58%)
  adding: lora_model/README.md (deflated 66%)
  adding: lora_model/adapter_model.safetensors (deflated 7%)
  adding: hp_ft_model/ (stored 0%)
  adding: hp_ft_model/model.safetensors (deflated 8%)
  adding: hp_ft_model/generation_config.json (deflated 29%)
  adding: hp_ft_model/config.json (deflated 63%)


In [76]:
# Install sentencepiece for T5/other tokenizers
!pip install sentencepiece

In [80]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import PeftModel
import torch
import os

device = "cuda" if torch.cuda.is_available() else "cpu"

# Base model
base_path = "/content/drive/MyDrive/all_models/base_model"
tokenizer = AutoTokenizer.from_pretrained(base_path)
base_model = AutoModelForSeq2SeqLM.from_pretrained(base_path).to(device)

# Full FT model
full_ft_path = "/content/drive/MyDrive/all_models/full_ft_model"
full_ft_model = AutoModelForSeq2SeqLM.from_pretrained(full_ft_path).to(device)

# LoRA adapter model
lora_path = "/content/drive/MyDrive/all_models/lora_model"
lora_model = PeftModel.from_pretrained(base_model, lora_path).to(device)

# HP-FT (normal fine-tune)
hp_ft_path = "/content/drive/MyDrive/all_models/hp_ft_model"
hp_model = AutoModelForSeq2SeqLM.from_pretrained(hp_ft_path).to(device)

# Dictionary of models
models = {
    "base_model": base_model,
    "full_ft_model": full_ft_model,
    "lora_model": lora_model,
    "hp_ft_model": hp_model
}

print("✅ All models loaded successfully!")

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

✅ All models loaded successfully!


In [2]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import PeftModel
import torch
from prettytable import PrettyTable

device = "cuda" if torch.cuda.is_available() else "cpu"

# Base model path (use this tokenizer for everything)
base_model_path = "/content/drive/MyDrive/all_models/base_model"

# Fine-tuned model paths (weights only)
model_paths = {
    "Base": base_model_path,
    "Full FT": "/content/drive/MyDrive/all_models/full_ft_model",
    "LoRA": "/content/drive/MyDrive/all_models/lora_model",
    "HP-FT": "/content/drive/MyDrive/all_models/hp_ft_model"
}

print("Loading base tokenizer ...")
tokenizer = AutoTokenizer.from_pretrained(base_model_path)

models = {}

print("Loading models ...")
for name, path in model_paths.items():
    if name == "LoRA":
        # LoRA needs the base model as backbone
        base_model = AutoModelForSeq2SeqLM.from_pretrained(base_model_path).to(device)
        model = PeftModel.from_pretrained(base_model, path).to(device)
    else:
        # Full weights or safetensors
        model = AutoModelForSeq2SeqLM.from_pretrained(path).to(device)
    model.eval()
    models[name] = model

print("✅ All models loaded!\n")

def generate_summary(model, text, max_new_tokens=100):
    prompt = "summarize: " + text.strip()
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, padding="longest").to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=4,
            no_repeat_ngram_size=3,
            length_penalty=1.0,
            early_stopping=True
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Example paragraph
paragraph = """Seoul, South Korea (CNN) -- North Korea is planning a nuclear test in the area where it has staged previous atomic blasts, according to a report from South Korean intelligence officials obtained by CNN.
The country appears to be pressing ahead with its planned rocket launch, raising concerns among neighboring countries and the United States. Experts warn that this may increase tensions in the region and prompt diplomatic talks."""

# Generate summaries
results = {}
for name, model in models.items():
    print(f"Generating summary with {name} ...")
    summary = generate_summary(model, paragraph)
    results[name] = summary

# Display table
table = PrettyTable()
table.field_names = ["Model", "Summary"]
for name, summary in results.items():
    table.add_row([name, summary])

print("\n=== Summary Comparison ===")
print(table)

Loading base tokenizer ...
Loading models ...


Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

✅ All models loaded!

Generating summary with Base ...
Generating summary with Full FT ...
Generating summary with LoRA ...
Generating summary with HP-FT ...

=== Summary Comparison ===
+---------+-----------------------------------+
|  Model  |              Summary              |
+---------+-----------------------------------+
|   Base  |                                   |
| Full FT |                                   |
|   LoRA  |                                   |
|  HP-FT  |                                   |
+---------+-----------------------------------+


In [1]:
from peft import PeftModel
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

base_model_path = "/content/drive/MyDrive/all_models/base_model"
lora_path = "/content/drive/MyDrive/all_models/lora_model"

# Load base model + tokenizer
tokenizer = AutoTokenizer.from_pretrained(base_model_path)
base_model = AutoModelForSeq2SeqLM.from_pretrained(base_model_path, device_map="auto")

# Load LoRA weights on top of base
lora_model = PeftModel.from_pretrained(base_model, lora_path)
lora_model.to(device)

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

NameError: name 'device' is not defined

In [3]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
from prettytable import PrettyTable

# 1️⃣ Device setup
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# 2️⃣ Paths to your models on Google Drive
model_paths = {
    "Base": "/content/drive/MyDrive/all_models/base_model",
    "Full FT": "/content/drive/MyDrive/all_models/full_ft_model",
    "LoRA": "/content/drive/MyDrive/all_models/lora_model",
    "HP-FT": "/content/drive/MyDrive/all_models/hp_ft_model"
}

# 3️⃣ Load base tokenizer once
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_paths["Base"])

# 4️⃣ Load models
models = {}
for name, path in model_paths.items():
    print(f"Loading {name} model ...")
    model = AutoModelForSeq2SeqLM.from_pretrained(path, device_map="auto")
    model.to(device)
    models[name] = model
print("✅ All models loaded!")

# 5️⃣ Example paragraph (CNN/DailyMail)
text = """Your CNN/DailyMail paragraph goes here."""

# 6️⃣ Generate summaries
results = {}
for name, model in models.items():
    print(f"Generating summary with {name} ...")
    inputs = tokenizer(text, return_tensors="pt", truncation=True).to(device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        num_beams=4,
        no_repeat_ngram_size=3,
        length_penalty=1.0,
        early_stopping=True
    )
    summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
    results[name] = summary

# 7️⃣ Display comparison table
table = PrettyTable()
table.field_names = ["Model", "Summary"]
for name, summary in results.items():
    table.add_row([name, summary])

print("\n=== Summary Comparison ===")
print(table)

Using device: cpu
Loading tokenizer...
Loading Base model ...


Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

Loading Full FT model ...


Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

Loading LoRA model ...


...

Loading weights: 0it [00:00, ?it/s]

T5ForConditionalGeneration LOAD REPORT from: /content/drive/MyDrive/all_models/lora_model
Key                                                                                     | Status     | 
----------------------------------------------------------------------------------------+------------+-
base_model.model.decoder.block.{0...11}.layer.0.SelfAttention.q.lora_A.default.weight   | UNEXPECTED | 
base_model.model.decoder.block.{0...11}.layer.0.SelfAttention.v.lora_B.default.weight   | UNEXPECTED | 
base_model.model.encoder.block.{0...11}.layer.0.SelfAttention.q.lora_B.default.weight   | UNEXPECTED | 
base_model.model.decoder.block.{0...11}.layer.1.EncDecAttention.v.lora_B.default.weight | UNEXPECTED | 
base_model.model.decoder.block.{0...11}.layer.1.EncDecAttention.q.lora_B.default.weight | UNEXPECTED | 
base_model.model.decoder.block.{0...11}.layer.1.EncDecAttention.q.lora_A.default.weight | UNEXPECTED | 
base_model.model.encoder.block.{0...11}.layer.0.SelfAttention.q.lora_A.default

Loading HP-FT model ...


Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

✅ All models loaded!
Generating summary with Base ...
Generating summary with Full FT ...
Generating summary with LoRA ...
Generating summary with HP-FT ...

=== Summary Comparison ===
+---------+--------------------------------------+
|  Model  |               Summary                |
+---------+--------------------------------------+
|   Base  |                                      |
| Full FT |                                      |
|   LoRA  |                                      |
|  HP-FT  |                                      |
+---------+--------------------------------------+
